## Imports & Connection

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.utils import resample
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [2]:
## Connection
connection = presto.connect(
        host='presto-gateway.processing.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Methodology 

1. Take hyderabad and analysis it for recent days
2. Success metric TTC, COBRA
3. Check the proportion of control and test split
    - 80 Control : 20 Test
    - Need proper sampling
4. Check the proportion on customer segment, order status and time temporal
    - Similar customer segment
    - Sweked order status and time temporal
        Need to make it as like to like in comparison group
5. Sampling 
    - Do sampling using service, order_status and time temporal level (Without exposing success metric)
    - Ensure sampled data behaving like-to-like 
6. Analysis 
    - Overall 
    - Service
    - Service x Customer segment 
    - Service x Customer segment x Time temporal

## Parameter & Datasets

In [3]:

# Define your list of cities and service names
cities = ['Hyderabad'] #'Bangalore', 'Delhi', 'Chennai', 'Jaipur', 'Pune', 'Kolkata']
service_names = ['CabEconomy'] #, 'Auto', 'CabEconomy', 'Link']

current_date = datetime.now()
start_date = '20240513'
end_date = '20240602'

# Convert date strings to datetime objects
start_dt = datetime.strptime(start_date, '%Y%m%d')
end_dt = datetime.strptime(end_date, '%Y%m%d')
segment_date = end_dt.strftime('%Y-%m-%d')
print(start_date, ' to ' ,end_date)

20240513  to  20240602


### Data

In [10]:
# Initialize an empty DataFrame to hold the results
df_data = pd.DataFrame()

# Loop through each city
for city in cities:
    # Loop through each service name
    for service_name in service_names:
        print(f"Processing for city: {city}, service name: {service_name}")

        # Loop through each date in the range
        current_dt = start_dt
        while current_dt <= end_dt:
            # Format the current date as a string in 'YYYYMMDD' format
            current_date_str = current_dt.strftime('%Y%m%d')
            
            data_query = f"""
            WITH olf_data AS (
                SELECT
                    yyyymmdd,
                    city_name,
                    service_obj_service_name service_name,
                    customer_id,
                    order_id,
                    order_status,
                    modified_order_status,
                    spd_fraud_flag,
                    CASE WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_to_cancel_in_sec,
                    CASE 
                        WHEN CAST(SUBSTR(quarter_hour,1,2) AS REAL) >= 8 AND CAST(SUBSTR(quarter_hour,1,2) AS REAL) <= 11 THEN 'morning_peak'
                        WHEN CAST(SUBSTR(quarter_hour,1,2) AS REAL) > 11 AND CAST(SUBSTR(quarter_hour,1,2) AS REAL) < 17 THEN 'afternoon'
                        WHEN CAST(SUBSTR(quarter_hour,1,2) AS REAL) >= 17 AND CAST(SUBSTR(quarter_hour,1,2) AS REAL) <= 21 THEN 'evening_peak'
                        WHEN CAST(SUBSTR(quarter_hour,1,2) AS REAL) >= 0 AND CAST(SUBSTR(quarter_hour,1,2) AS REAL) < 8 THEN 'rest_morning'
                        ELSE 'rest_evening' 
                    END AS time_period,
                    accept_to_pickup_distance fm_distance,
                    distance_final_distance lm_distance,
                    pickup_cluster
                FROM
                    orders.order_logs_fact
                WHERE 
                    yyyymmdd = '{current_date_str}'
                    AND city_name = '{city}'
                    AND service_obj_service_name = '{service_name}'
            ),
            orderLogsFact AS (
                SELECT
                    yyyymmdd,
                    customer_id,
                    city_name,
                    service_name
                FROM
                    olf_data
                GROUP BY 
                    1,2,3,4
            ),
            intervention AS (
                SELECT 
                    yyyymmdd,
                    userId,
                    group_tc,
                    MAX(COALESCE(dispatch_ad_request, dispatch_ad_response, dispatch_ad_viewable_impression, dispatch_ad_click)) mapping_v1,
                    MAX(COALESCE(dispatch_ad_response, dispatch_ad_viewable_impression, dispatch_ad_click)) mapping_v2,
                    MAX(COALESCE(dispatch_ad_viewable_impression, dispatch_ad_click)) mapping_v3
                FROM
                    (
                    SELECT 
                        yyyymmdd,
                        userId,
                        'test' group_tc,
                        CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Request' THEN yyyymmdd || '-' || userId || '-' || hhmmss END AS dispatch_ad_request,
                        CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Response' THEN yyyymmdd || '-' || userId || '-' || hhmmss END AS dispatch_ad_response,
                        CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Viewable_Impression' THEN yyyymmdd || '-' || userId || '-' || hhmmss END AS dispatch_ad_viewable_impression,
                        CASE WHEN pagename = 'CaptainSearchScreen' AND eventName = 'Ad_Click' THEN yyyymmdd || '-' || userId || '-' || hhmmss END AS dispatch_ad_click
                    FROM 
                        canonical.iceberg_log_telemetry_ads_impressions_immutable
                    WHERE  
                        yyyymmdd = '{current_date_str}'
                        AND responseType = 'GAMBanner'
                        AND pagename = 'CaptainSearchScreen'
                        AND city = '{city}'
                        AND service = '{service_name}'
                        AND userId IN (SELECT DISTINCT customer_id FROM orderLogsFact)
                    )
                GROUP BY 1,2,3
            ),
            customer_segment AS (
                SELECT 
                    customer_id,
                    taxi_lifetime_stage customer_segment
                FROM
                    datasets.iallocator_customer_segments
                WHERE 
                    run_date = '{segment_date}'
                    AND taxi_recency_segment != 'INACTIVE'
            ),
            experiment_base AS (
                SELECT
                    a.yyyymmdd,
                    a.city_name,
                    a.customer_id,
                    COALESCE(b.group_tc, 'control') group_tc,
                    customer_segment
                FROM
                    orderLogsFact a
                LEFT JOIN 
                    intervention b ON a.yyyymmdd = b.yyyymmdd AND a.customer_id = b.userId
                LEFT JOIN 
                    customer_segment c ON a.customer_id = c.customer_id
                GROUP BY 1,2,3,4,5
            )
            SELECT
                a.yyyymmdd,
                a.city_name,
                a.customer_id,
                a.group_tc,
                COALESCE(a.customer_segment, 'UNKNOWN') customer_segment,
                b.service_name,
                b.order_id,
                b.order_status,
                b.modified_order_status,
                b.time_period,
                b.fm_distance,
                b.lm_distance,
                pickup_cluster,
                COUNT(DISTINCT b.order_id) gross_orders,
                COUNT(DISTINCT CASE WHEN b.order_status = 'dropped' AND spd_fraud_flag != TRUE THEN b.order_id END) net_orders,
                COUNT(DISTINCT CASE WHEN b.modified_order_status = 'COBRM' THEN b.order_id END) cobrm,
                COUNT(DISTINCT CASE WHEN b.modified_order_status = 'COBRA' THEN b.order_id END) cobra,
                COUNT(DISTINCT CASE WHEN b.modified_order_status = 'OCARA' THEN b.order_id END) ocara,
                COUNT(DISTINCT CASE WHEN b.modified_order_status = 'aborted' THEN b.order_id END) aborted,
                COUNT(DISTINCT CASE WHEN b.modified_order_status = 'expired' THEN b.order_id END) expired,
                AVG(CASE WHEN b.modified_order_status = 'COBRM' THEN time_to_cancel_in_sec END) AS avg_cobrm_time,
                AVG(CASE WHEN b.modified_order_status = 'COBRA' THEN time_to_cancel_in_sec END) AS avg_cobra_time,
                AVG(CASE WHEN b.modified_order_status = 'OCARA' THEN time_to_cancel_in_sec END) AS avg_ocara_time
            FROM 
                experiment_base a
            JOIN 
                olf_data b ON a.yyyymmdd = b.yyyymmdd AND a.customer_id = b.customer_id AND a.city_name = b.city_name
            WHERE 
                LOWER(modified_order_status) NOT IN ('ontheway', 'started', 'new', 'arrived', 'requested')
            GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13
            """
            
            # Execute the query and get the result as a DataFrame
            df_day = pd.read_sql(data_query, connection)
            
            # Concatenate the result with the existing DataFrame
            df_data = pd.concat([df_data, df_day], ignore_index=True)
            
            # Move to the next date
            current_dt += timedelta(days=1)

# df_data.to_csv('hyderabad_auto_weekly_v1.csv', index=False)
df_data.to_csv('hyderabad_cab_weekly_v1.csv', index=False)

Processing for city: Hyderabad, service name: CabEconomy


In [4]:
# df_data.to_csv('hyderabad_weekly_v1.csv', index=False)

df_data_auto = pd.read_csv('hyderabad_auto_weekly_v1.csv')
df_data_link = pd.read_csv('hyderabad_link_weekly_v1.csv')
df_data_cab = pd.read_csv('hyderabad_cab_weekly_v1.csv')

df_data = pd.concat([df_data_link, df_data_auto, df_data_cab])

In [5]:
mapping = {
    'rest_morning': '5. rest_morning',
    'morning_peak': '1. morning_peak',
    'afternoon': '2. afternoon',
    'evening_peak': '3. evening_peak',
    'rest_evening': '4. rest_evening'
}

# Apply the mapping to the time_period column
df_data['time_period'] = df_data['time_period'].map(mapping)

In [6]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11459627 entries, 0 to 1065024
Data columns (total 23 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   yyyymmdd               int64  
 1   city_name              object 
 2   customer_id            object 
 3   group_tc               object 
 4   customer_segment       object 
 5   service_name           object 
 6   order_id               object 
 7   order_status           object 
 8   modified_order_status  object 
 9   time_period            object 
 10  fm_distance            float64
 11  lm_distance            float64
 12  pickup_cluster         object 
 13  gross_orders           int64  
 14  net_orders             int64  
 15  cobrm                  int64  
 16  cobra                  int64  
 17  ocara                  int64  
 18  aborted                int64  
 19  expired                int64  
 20  avg_cobrm_time         float64
 21  avg_cobra_time         float64
 22  avg_ocara_time        

In [7]:
int_columns = df_data.select_dtypes(include=['int64']).columns.tolist()
df_data[int_columns] = df_data[int_columns].astype('int32')
df_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11459627 entries, 0 to 1065024
Data columns (total 23 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   yyyymmdd               int32  
 1   city_name              object 
 2   customer_id            object 
 3   group_tc               object 
 4   customer_segment       object 
 5   service_name           object 
 6   order_id               object 
 7   order_status           object 
 8   modified_order_status  object 
 9   time_period            object 
 10  fm_distance            float64
 11  lm_distance            float64
 12  pickup_cluster         object 
 13  gross_orders           int32  
 14  net_orders             int32  
 15  cobrm                  int32  
 16  cobra                  int32  
 17  ocara                  int32  
 18  aborted                int32  
 19  expired                int32  
 20  avg_cobrm_time         float64
 21  avg_cobra_time         float64
 22  avg_ocara_time        

## Check biasness

In [8]:
df_data['group_tc'].value_counts(True)

group_tc
test       0.542301
control    0.457699
Name: proportion, dtype: float64

### customer_segment
Similar customer_segment proportion

In [9]:
df_data[df_data['group_tc'] == 'test']['customer_segment'].value_counts(True)

customer_segment
COMMITTED      0.453734
HOOK           0.263076
SUSTENANCE     0.111807
HANDHOLDING    0.068252
CHURN_OTB      0.035164
DORMANT        0.030241
UNKNOWN        0.018789
SOFT_CHURN     0.015552
DETOX          0.003385
Name: proportion, dtype: float64

In [10]:
df_data[df_data['group_tc'] == 'control']['customer_segment'].value_counts(True)

customer_segment
COMMITTED      0.441140
HOOK           0.231514
SUSTENANCE     0.114296
HANDHOLDING    0.062038
CHURN_OTB      0.051954
DORMANT        0.043503
SOFT_CHURN     0.027428
UNKNOWN        0.024149
DETOX          0.003978
Name: proportion, dtype: float64

### order_status
Not similar order_status proportion

In [11]:
df_data[df_data['group_tc'] == 'test']['order_status'].value_counts(True)

order_status
dropped              0.637319
customerCancelled    0.305818
expired              0.055371
aborted              0.001492
Name: proportion, dtype: float64

In [12]:
df_data[df_data['group_tc'] == 'control']['order_status'].value_counts(True)

order_status
dropped              0.611918
customerCancelled    0.318508
expired              0.067867
aborted              0.001707
Name: proportion, dtype: float64

### modified_order_status
Not similar modified_order_status proportion

In [13]:
df_data[df_data['group_tc'] == 'test']['modified_order_status'].value_counts(True)

modified_order_status
dropped    0.637319
OCARA      0.155996
COBRA      0.144826
expired    0.055371
COBRM      0.004996
aborted    0.001492
Name: proportion, dtype: float64

In [14]:
df_data[df_data['group_tc'] == 'control']['modified_order_status'].value_counts(True)

modified_order_status
dropped    0.611918
COBRA      0.158348
OCARA      0.155554
expired    0.067867
COBRM      0.004606
aborted    0.001707
Name: proportion, dtype: float64

### time_period
Not similar time_period proportion

In [15]:
df_data[df_data['group_tc'] == 'test']['time_period'].value_counts(True)

time_period
3. evening_peak    0.336049
2. afternoon       0.287880
1. morning_peak    0.228248
5. rest_morning    0.080457
4. rest_evening    0.067366
Name: proportion, dtype: float64

In [16]:
df_data[df_data['group_tc'] == 'control']['time_period'].value_counts(True)

time_period
3. evening_peak    0.317155
2. afternoon       0.268184
1. morning_peak    0.236039
5. rest_morning    0.108549
4. rest_evening    0.070073
Name: proportion, dtype: float64

### pickup_cluster


In [17]:
# df_data[df_data['group_tc'] == 'test']['pickup_cluster'].value_counts(True)

In [18]:
# df_data[df_data['group_tc'] == 'control']['pickup_cluster'].value_counts(True)

## User Definition

In [19]:
df_data['sampling_column'] = df_data['customer_segment'] + '-' + df_data['time_period'] + '-' + df_data['modified_order_status']
df_data['sampling_column'] = df_data['sampling_column'].str.lower()


## accept to pickup - fm distance
df_data['fm_bucket'] = np.where(df_data['fm_distance'] <= 1, 'Less', 
                               np.where(df_data['fm_distance'] <= 2, 'Medium', 
                                        np.where(df_data['fm_distance'] > 2, 'High', 'no-fm')))

In [20]:
customer_agg_dict = {
        'customers': ('customer_id', 'nunique'),
        'gross_orders': ('gross_orders', 'sum'),
        'net_orders': ('net_orders', 'sum'),
        'cobrm': ('cobrm', 'sum'),
        'cobra': ('cobra', 'sum'),
        'ocara': ('ocara', 'sum'),
        'aborted': ('aborted', 'sum'),
        'expired': ('expired', 'sum'),
        'cobrm_time': ('avg_cobrm_time', 'median'),
        'cobra_time': ('avg_cobra_time', 'median'),
        'ocara_time': ('avg_ocara_time', 'median')
}

def calculate_percentage(df, numerator, denominator):
    percentage = (df[numerator] * 100.00 / df[denominator]).round(2)
    return percentage

def calculate_percentage_columns(df):
        
    df['g2n'] = calculate_percentage(df, 'net_orders', 'gross_orders')
    df['% cobrm'] = calculate_percentage(df, 'cobrm', 'gross_orders')
    df['% cobra'] = calculate_percentage(df, 'cobra', 'gross_orders')
    df['% ocara'] = calculate_percentage(df, 'ocara', 'gross_orders')
    df['% expired'] = calculate_percentage(df, 'expired', 'gross_orders')
    df['% aborted'] = calculate_percentage(df, 'aborted', 'gross_orders')
    
    df_new = df[['group_tc', 'g2n', '% cobrm', '% cobra', '% ocara', '% expired', '% aborted',
                 'cobrm_time', 'cobra_time', 'ocara_time' ]] 
    
    return df_new

In [21]:
df_data.columns

Index(['yyyymmdd', 'city_name', 'customer_id', 'group_tc', 'customer_segment',
       'service_name', 'order_id', 'order_status', 'modified_order_status',
       'time_period', 'fm_distance', 'lm_distance', 'pickup_cluster',
       'gross_orders', 'net_orders', 'cobrm', 'cobra', 'ocara', 'aborted',
       'expired', 'avg_cobrm_time', 'avg_cobra_time', 'avg_ocara_time',
       'sampling_column', 'fm_bucket'],
      dtype='object')

In [22]:
def sampling(df):
    df_data = df.copy()
    
#     exposed_columns = ['yyyymmdd', 'group_tc', 'city_name', 'service_name', 
#                        #'customer_id', 
#                        'customer_segment', 'order_id', 'order_status', 'modified_order_status',
#                        'time_period', 'fm_bucket', 'lm_distance', 'sampling_column']
    
    # Filter to include only the necessary columns
    df_data = df_data #[exposed_columns]

    # Determine the stratify columns
    stratify_cols = ['service_name', 'time_period', 'modified_order_status']

    # Perform stratified sampling on the entire dataframe
    unique_combinations = df_data[stratify_cols].drop_duplicates()
    total_size = len(df_data)
    min_size_per_combination = total_size // len(unique_combinations)
    
    sampled_df = df_data.groupby(stratify_cols, group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), min_size_per_combination), random_state=1)
    ).reset_index(drop=True)

    # Split the sampled data into test and control groups
    test_sample = sampled_df[sampled_df['group_tc'] == 'test']
    control_sample = sampled_df[sampled_df['group_tc'] == 'control']

    # Ensure 99% similarity in row counts
    test_size = len(test_sample)
    control_size = len(control_sample)
    target_size = int(min(test_size, control_size) * 0.99)

    if test_size > target_size:
        test_sample = test_sample.sample(n=target_size, random_state=1)
    if control_size > target_size:
        control_sample = control_sample.sample(n=target_size, random_state=1)

    # Ensure proportions are maintained
    def check_proportions(df, column):
        return df[column].value_counts(normalize=True)

    test_proportions = check_proportions(test_sample, 'modified_order_status')
    control_proportions = check_proportions(control_sample, 'modified_order_status')

    print("Test group proportions:\n", test_proportions)
    print("Control group proportions:\n", control_proportions)

    balanced_df = pd.concat([test_sample, control_sample]).reset_index(drop=True)
    
    return balanced_df

In [23]:
iterations = 3

def iterative_sampling(df, iterations):
    balanced_df = df
    for i in range(iterations):
        balanced_df = sampling(balanced_df)
    return balanced_df

balanced_df = iterative_sampling(df_data, iterations)

Test group proportions:
 modified_order_status
dropped    0.361297
COBRA      0.250553
OCARA      0.246839
expired    0.125277
COBRM      0.012343
aborted    0.003692
Name: proportion, dtype: float64
Control group proportions:
 modified_order_status
dropped    0.354583
COBRA      0.254542
OCARA      0.237774
expired    0.139530
COBRM      0.009894
aborted    0.003676
Name: proportion, dtype: float64
Test group proportions:
 modified_order_status
dropped    0.290173
COBRA      0.256107
OCARA      0.254781
expired    0.170827
COBRM      0.021639
aborted    0.006473
Name: proportion, dtype: float64
Control group proportions:
 modified_order_status
dropped    0.274383
COBRA      0.269078
OCARA      0.241608
expired    0.192571
COBRM      0.016282
aborted    0.006079
Name: proportion, dtype: float64
Test group proportions:
 modified_order_status
dropped    0.266096
OCARA      0.251680
COBRA      0.247674
expired    0.190961
COBRM      0.033542
aborted    0.010046
Name: proportion, dtype: fl

In [24]:
balanced_df.service_name.nunique()

3

In [25]:
df_data.service_name.unique()

array(['Link', 'Auto', 'CabEconomy'], dtype=object)

In [26]:
df_sampled = balanced_df

In [27]:
df_sampled.head(2)

,yyyymmdd,city_name,customer_id,group_tc,customer_segment,service_name,order_id,order_status,modified_order_status,time_period,fm_distance,lm_distance,pickup_cluster,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,avg_cobrm_time,avg_cobra_time,avg_ocara_time,sampling_column,fm_bucket
0,20240522,Hyderabad,5d23887b06ee2c37f1ed7232,test,SOFT_CHURN,CabEconomy,664e1462c91b4000d1668b46,dropped,dropped,3. evening_peak,0.424,13.510,Falaknuma,1,1,0,0,0,0,0,NaN,NaN,NaN,soft_churn-3. evening_peak-dropped,Less
1,20240516,Hyderabad,6240396459c22152a6621bed,test,UNKNOWN,Auto,6645f8d36d7520329153ad6b,expired,expired,3. evening_peak,NaN,9.306,Telecom Nagar Gachibowli,1,0,0,0,0,0,1,NaN,NaN,NaN,unknown-3. evening_peak-expired,no-fm


In [28]:
df_test = df_sampled\
.groupby(['city_name', 
#           'fm_bucket',
#           'service_name',
#           'customer_segment',
#           'time_period',
          'group_tc'])\
.agg(**customer_agg_dict)\
.reset_index()


df_test['g2n'] = calculate_percentage(df_test, 'net_orders', 'gross_orders')
df_test['% cobrm'] = calculate_percentage(df_test, 'cobrm', 'gross_orders')
df_test['% cobra'] = calculate_percentage(df_test, 'cobra', 'gross_orders')
df_test['% ocara'] = calculate_percentage(df_test, 'ocara', 'gross_orders')
df_test['% expired'] = calculate_percentage(df_test, 'expired', 'gross_orders')
df_test['% aborted'] = calculate_percentage(df_test, 'aborted', 'gross_orders')


df_test[['city_name', 
#          'fm_bucket',
#          'service_name', 
#          'time_period', 
#          'customer_segment',
         'group_tc','customers', 'gross_orders', 'net_orders', 
         'g2n', '% cobrm', '% cobra', '% ocara', '% expired', '% aborted',
         'cobrm_time', 'cobra_time', 'ocara_time']]

,city_name,group_tc,customers,gross_orders,net_orders,g2n,% cobrm,% cobra,% ocara,% expired,% aborted,cobrm_time,cobra_time,ocara_time
0,Hyderabad,control,530820,870538,210457,24.18,2.46,26.11,24.15,22.04,0.92,95.0,122.0,314.0
1,Hyderabad,test,512295,870538,230508,26.48,3.35,24.77,25.17,19.10,1.00,79.0,128.0,321.0


In [29]:
df_test = df_data\
.groupby(['city_name', 
#           'fm_bucket',
#           'service_name',
#           'customer_segment',
#           'time_period',
          'group_tc'])\
.agg(**customer_agg_dict)\
.reset_index()


df_test['g2n'] = calculate_percentage(df_test, 'net_orders', 'gross_orders')
df_test['% cobrm'] = calculate_percentage(df_test, 'cobrm', 'gross_orders')
df_test['% cobra'] = calculate_percentage(df_test, 'cobra', 'gross_orders')
df_test['% ocara'] = calculate_percentage(df_test, 'ocara', 'gross_orders')
df_test['% expired'] = calculate_percentage(df_test, 'expired', 'gross_orders')
df_test['% aborted'] = calculate_percentage(df_test, 'aborted', 'gross_orders')


df_test[['city_name', 
#          'fm_bucket',
#          'service_name', 
#          'time_period', 
#          'customer_segment',
         'group_tc','customers', 'gross_orders', 'net_orders', 
         'g2n', '% cobrm', '% cobra', '% ocara', '% expired', '% aborted',
         'cobrm_time', 'cobra_time', 'ocara_time']]

,city_name,group_tc,customers,gross_orders,net_orders,g2n,% cobrm,% cobra,% ocara,% expired,% aborted,cobrm_time,cobra_time,ocara_time
0,Hyderabad,control,1689461,5245058,3185881,60.74,0.46,15.83,15.56,6.79,0.17,94.0,123.0,307.0
1,Hyderabad,test,1627565,6214569,3938309,63.37,0.50,14.48,15.60,5.54,0.15,79.0,131.0,318.0


## Analysis

In [30]:
df_data.columns

Index(['yyyymmdd', 'city_name', 'customer_id', 'group_tc', 'customer_segment',
       'service_name', 'order_id', 'order_status', 'modified_order_status',
       'time_period', 'fm_distance', 'lm_distance', 'pickup_cluster',
       'gross_orders', 'net_orders', 'cobrm', 'cobra', 'ocara', 'aborted',
       'expired', 'avg_cobrm_time', 'avg_cobra_time', 'avg_ocara_time',
       'sampling_column', 'fm_bucket'],
      dtype='object')

In [71]:
df_data.groupby(['yyyymmdd','group_tc']).agg({'customer_id' : 'nunique'}).reset_index()

,yyyymmdd,group_tc,customer_id
0,2024-05-13,control,172473
1,2024-05-13,test,65918
2,2024-05-14,control,270136
3,2024-05-14,test,93610
4,2024-05-15,control,252787
5,2024-05-15,test,92679
6,2024-05-16,control,268522
7,2024-05-16,test,122262
8,2024-05-17,control,231019
9,2024-05-17,test,94916


In [32]:
df_data_filter = df_data[df_data['yyyymmdd'].isin(['20240520','20240521','20240522','20240523',
                                                   '20240524','20240525','20240526','20240527',
                                                   '20240528','20240529'])]

### Overall

In [33]:
# df_data['yyyymmdd'] = pd.to_datetime(df_data['yyyymmdd'], format='%Y%m%d')
# df_data['start_date_week'] = df_data['yyyymmdd'].apply(lambda x: x - pd.Timedelta(days=x.weekday()))

#### City

In [50]:
grouper_column = ['city_name', 'group_tc']

df_agg_city = df_data_filter\
.groupby(grouper_column)\
.agg(**customer_agg_dict)\
.sort_values(by=grouper_column)\
.reset_index()

df_agg_city

,city_name,group_tc,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time
0,Hyderabad,control,828753,1986727,1323193,7384,231221,323444,3593,88520,93.0,115.0,295.0
1,Hyderabad,test,1101277,3371458,2251265,17836,404636,532393,5289,146497,80.0,124.0,314.0


In [51]:
df_agg_city_percentage_columns = calculate_percentage_columns(df_agg_city)
df_agg_city_percentage_columns

,group_tc,g2n,% cobrm,% cobra,% ocara,% expired,% aborted,cobrm_time,cobra_time,ocara_time
0,control,66.60,0.37,11.64,16.28,4.46,0.18,93.0,115.0,295.0
1,test,66.77,0.53,12.00,15.79,4.35,0.16,80.0,124.0,314.0


In [53]:
df_agg_city_percentage_columns.to_clipboard(index=False)

#### City & Service

In [54]:
grouper_column = ['city_name', 'service_name', 'group_tc']

df_agg_city_service_name = df_data_filter\
.groupby(grouper_column)\
.agg(**customer_agg_dict)\
.sort_values(by=grouper_column)\
.reset_index()

df_agg_city_service_name

,city_name,service_name,group_tc,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time
0,Hyderabad,Auto,control,334731,750403,497880,1786,79087,132347,1090,36406,77.0,125.0,274.0
1,Hyderabad,Auto,test,715880,1850357,1231261,7562,208146,308072,2426,88160,41.0,133.0,308.0
2,Hyderabad,CabEconomy,control,127328,221083,122410,869,40215,38602,232,18742,107.0,119.0,325.0
3,Hyderabad,CabEconomy,test,131597,248721,132222,1668,52593,41880,298,20051,103.0,126.0,347.0
4,Hyderabad,Link,control,495880,1015241,702903,4729,111919,152495,2271,33372,94.0,107.0,307.0
5,Hyderabad,Link,test,490975,1272380,887782,8606,143897,182441,2565,38286,98.0,111.0,318.0


In [55]:
df_agg_city_service_percentage_columns = calculate_percentage_columns(df_agg_city_service_name)
df_agg_city_service_percentage_columns

,group_tc,g2n,% cobrm,% cobra,% ocara,% expired,% aborted,cobrm_time,cobra_time,ocara_time
0,control,66.35,0.24,10.54,17.64,4.85,0.15,77.0,125.0,274.0
1,test,66.54,0.41,11.25,16.65,4.76,0.13,41.0,133.0,308.0
2,control,55.37,0.39,18.19,17.46,8.48,0.10,107.0,119.0,325.0
3,test,53.16,0.67,21.15,16.84,8.06,0.12,103.0,126.0,347.0
4,control,69.24,0.47,11.02,15.02,3.29,0.22,94.0,107.0,307.0
5,test,69.77,0.68,11.31,14.34,3.01,0.20,98.0,111.0,318.0


In [57]:
df_agg_city_service_percentage_columns.to_clipboard(index=False)

#### City & Time period

In [38]:
grouper_column = ['city_name', 'time_period', 'group_tc']

df_agg_city_time_period = df_data_filter\
.groupby(grouper_column)\
.agg(**customer_agg_dict)\
.sort_values(by=grouper_column)\
.reset_index()

df_agg_city_time_period

,city_name,time_period,group_tc,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time
0,Hyderabad,1. morning_peak,control,276374,500684,334531,1461,66503,70322,685,25347,94.0,122.0,289.0
1,Hyderabad,1. morning_peak,test,402160,831595,566154,3057,107962,114023,938,36756,75.0,128.0,308.0
2,Hyderabad,2. afternoon,control,328405,551384,388639,1433,51852,88096,1057,17858,87.0,110.0,301.0
3,Hyderabad,2. afternoon,test,507791,966983,686017,4381,89349,148810,1554,33082,72.0,119.0,319.0
4,Hyderabad,3. evening_peak,control,344297,616299,401061,2586,72684,106912,1237,28765,89.5,111.0,297.0
5,Hyderabad,3. evening_peak,test,532029,1087324,699989,6087,139759,183847,1942,51050,76.0,124.0,322.0
6,Hyderabad,4. rest_evening,control,86369,124936,70825,485,20515,23173,327,8792,98.0,118.0,308.0
7,Hyderabad,4. rest_evening,test,130957,211369,113225,1702,41423,37301,494,16163,85.0,128.0,323.0
8,Hyderabad,5. rest_morning,control,135696,193424,128137,1419,19667,34941,287,7758,99.0,112.0,278.0
9,Hyderabad,5. rest_morning,test,176753,274187,185880,2609,26143,48412,361,9446,101.0,119.0,279.0


In [39]:
df_agg_city_time_period_percentage_columns = calculate_percentage_columns(df_agg_city_time_period)
df_agg_city_time_period_percentage_columns

,group_tc,g2n,% cobrm,% cobra,% ocara,% expired,% aborted,cobrm_time,cobra_time,ocara_time
0,control,66.81,0.29,13.28,14.05,5.06,0.14,94.0,122.0,289.0
1,test,68.08,0.37,12.98,13.71,4.42,0.11,75.0,128.0,308.0
2,control,70.48,0.26,9.40,15.98,3.24,0.19,87.0,110.0,301.0
3,test,70.94,0.45,9.24,15.39,3.42,0.16,72.0,119.0,319.0
4,control,65.08,0.42,11.79,17.35,4.67,0.20,89.5,111.0,297.0
5,test,64.38,0.56,12.85,16.91,4.70,0.18,76.0,124.0,322.0
6,control,56.69,0.39,16.42,18.55,7.04,0.26,98.0,118.0,308.0
7,test,53.57,0.81,19.60,17.65,7.65,0.23,85.0,128.0,323.0
8,control,66.25,0.73,10.17,18.06,4.01,0.15,99.0,112.0,278.0
9,test,67.79,0.95,9.53,17.66,3.45,0.13,101.0,119.0,279.0


In [59]:
df_agg_city_time_period_percentage_columns.to_clipboard(index=False)

#### Segement

In [64]:
grouper_column = ['city_name', 'customer_segment', 'group_tc']

df_agg_city_service_segment = df_data_filter\
.groupby(grouper_column)\
.agg(**customer_agg_dict)\
.sort_values(by=grouper_column)\
.reset_index()

df_agg_city_service_segment

,city_name,customer_segment,group_tc,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time
0,Hyderabad,CHURN_OTB,control,44811,73734,54574,146,6671,9742,86,2247,97.5,115.0,310.0
1,Hyderabad,CHURN_OTB,test,57982,107772,79544,264,10117,14224,120,3167,85.5,126.0,319.5
2,Hyderabad,COMMITTED,control,212997,894731,615472,2895,103619,129186,1173,38897,88.0,112.0,268.0
3,Hyderabad,COMMITTED,test,288003,1600170,1107685,6444,191299,222710,1922,64389,64.0,118.0,297.0
4,Hyderabad,DETOX,control,2580,7542,5030,46,1135,942,5,344,85.5,110.0,261.5
5,Hyderabad,DETOX,test,3171,12958,8463,36,2255,1554,15,590,72.5,121.0,248.0
6,Hyderabad,DORMANT,control,44974,64382,0,696,20360,33589,525,8101,90.0,113.0,322.0
7,Hyderabad,DORMANT,test,51582,79832,0,1122,26352,40151,543,10592,88.0,125.0,318.0
8,Hyderabad,HANDHOLDING,control,92377,130489,96875,471,10431,17406,195,4560,105.0,122.0,351.0
9,Hyderabad,HANDHOLDING,test,123742,226567,153668,1254,22348,37944,463,9847,96.0,137.0,369.0


In [65]:
df_agg_city_service_segment_percentage_columns = calculate_percentage_columns(df_agg_city_service_segment)
df_agg_city_service_segment_percentage_columns

,group_tc,g2n,% cobrm,% cobra,% ocara,% expired,% aborted,cobrm_time,cobra_time,ocara_time
0,control,74.01,0.20,9.05,13.21,3.05,0.12,97.5,115.0,310.0
1,test,73.81,0.24,9.39,13.20,2.94,0.11,85.5,126.0,319.5
2,control,68.79,0.32,11.58,14.44,4.35,0.13,88.0,112.0,268.0
3,test,69.22,0.40,11.95,13.92,4.02,0.12,64.0,118.0,297.0
4,control,66.69,0.61,15.05,12.49,4.56,0.07,85.5,110.0,261.5
5,test,65.31,0.28,17.40,11.99,4.55,0.12,72.5,121.0,248.0
6,control,0.00,1.08,31.62,52.17,12.58,0.82,90.0,113.0,322.0
7,test,0.00,1.41,33.01,50.29,13.27,0.68,88.0,125.0,318.0
8,control,74.24,0.36,7.99,13.34,3.49,0.15,105.0,122.0,351.0
9,test,67.82,0.55,9.86,16.75,4.35,0.20,96.0,137.0,369.0


In [67]:
df_agg_city_service_segment_percentage_columns.to_clipboard(index=False)

#### City <> Service <> Time period

In [60]:
grouper_column = ['city_name', 'service_name', 'time_period', 'group_tc']

df_agg_city_service_time_period = df_data_filter\
.groupby(grouper_column)\
.agg(**customer_agg_dict)\
.sort_values(by=grouper_column)\
.reset_index()

df_agg_city_service_time_period

,city_name,service_name,time_period,group_tc,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time
0,Hyderabad,Auto,1. morning_peak,control,100584,174005,117220,291,18998,28552,202,8421,52.0,130.0,268.0
1,Hyderabad,Auto,1. morning_peak,test,231904,427345,294398,1228,47239,65093,406,18023,26.5,135.0,300.0
2,Hyderabad,Auto,2. afternoon,control,138572,223425,164216,375,13777,37555,339,6568,39.0,114.0,262.0
3,Hyderabad,Auto,2. afternoon,test,319854,554743,405347,2152,37371,88208,771,19448,28.0,127.0,301.0
4,Hyderabad,Auto,3. evening_peak,control,148138,259154,159903,646,32958,48773,451,15764,82.5,126.0,289.0
5,Hyderabad,Auto,3. evening_peak,test,337797,636956,394440,2640,87302,113154,975,36652,38.5,135.0,326.0
6,Hyderabad,Auto,4. rest_evening,control,29188,40672,20615,141,8227,7926,59,3589,90.0,129.0,286.0
7,Hyderabad,Auto,4. rest_evening,test,69649,104628,51224,575,23740,19215,182,9453,55.0,136.0,321.0
8,Hyderabad,Auto,5. rest_morning,control,40046,53147,35926,333,5127,9541,39,2064,100.0,127.0,247.0
9,Hyderabad,Auto,5. rest_morning,test,90145,126685,85852,967,12494,22402,92,4584,87.0,136.0,260.0


In [61]:
df_agg_city_service_time_period_percentage_columns = calculate_percentage_columns(df_agg_city_service_time_period)
df_agg_city_service_time_period_percentage_columns

,group_tc,g2n,% cobrm,% cobra,% ocara,% expired,% aborted,cobrm_time,cobra_time,ocara_time
0,control,67.37,0.17,10.92,16.41,4.84,0.12,52.0,130.0,268.0
1,test,68.89,0.29,11.05,15.23,4.22,0.10,26.5,135.0,300.0
2,control,73.50,0.17,6.17,16.81,2.94,0.15,39.0,114.0,262.0
3,test,73.07,0.39,6.74,15.90,3.51,0.14,28.0,127.0,301.0
4,control,61.70,0.25,12.72,18.82,6.08,0.17,82.5,126.0,289.0
5,test,61.93,0.41,13.71,17.76,5.75,0.15,38.5,135.0,326.0
6,control,50.69,0.35,20.23,19.49,8.82,0.15,90.0,129.0,286.0
7,test,48.96,0.55,22.69,18.37,9.03,0.17,55.0,136.0,321.0
8,control,67.60,0.63,9.65,17.95,3.88,0.07,100.0,127.0,247.0
9,test,67.77,0.76,9.86,17.68,3.62,0.07,87.0,136.0,260.0


In [63]:
df_agg_city_service_time_period_percentage_columns.to_clipboard(index=False)

### Daily trend

In [42]:
grouper_column = ['yyyymmdd', 'city_name', 'group_tc']

df_daily_city = df_data_filter\
.groupby(grouper_column)\
.agg(**customer_agg_dict)\
.sort_values(by=grouper_column)\
.reset_index()

df_daily_city

,yyyymmdd,city_name,group_tc,customers,gross_orders,net_orders,cobrm,cobra,ocara,aborted,expired,cobrm_time,cobra_time,ocara_time
0,2024-05-20,Hyderabad,control,208126,319982,206381,1341,43661,50753,551,15632,87.0,118.0,309.0
1,2024-05-20,Hyderabad,test,152054,249817,164227,1325,30317,41136,349,11552,78.0,128.0,307.0
2,2024-05-21,Hyderabad,control,169014,257307,171009,930,30874,40837,474,11908,98.0,114.0,300.0
3,2024-05-21,Hyderabad,test,162860,272710,180422,1484,33455,42997,431,12912,76.0,126.0,314.0
4,2024-05-22,Hyderabad,control,147766,225791,150537,777,26578,35655,392,10752,90.0,117.0,296.0
5,2024-05-22,Hyderabad,test,187273,315606,209834,1721,38744,49042,438,14644,81.0,124.0,312.0
6,2024-05-23,Hyderabad,control,128381,193050,134101,754,18848,31205,313,6862,91.0,109.0,286.0
7,2024-05-23,Hyderabad,test,194868,320802,222070,1741,33284,50285,467,11633,80.0,120.0,305.0
8,2024-05-24,Hyderabad,control,120532,181529,124487,599,18585,29901,325,6810,101.0,113.0,288.0
9,2024-05-24,Hyderabad,test,208919,343727,236328,1675,37650,53508,523,12615,73.0,120.0,309.0


In [43]:
df_daily_city_percentage_columns = calculate_percentage_columns(df_daily_city)
df_daily_city_percentage_columns

,group_tc,g2n,% cobrm,% cobra,% ocara,% expired,% aborted,cobrm_time,cobra_time,ocara_time
0,control,64.50,0.42,13.64,15.86,4.89,0.17,87.0,118.0,309.0
1,test,65.74,0.53,12.14,16.47,4.62,0.14,78.0,128.0,307.0
2,control,66.46,0.36,12.00,15.87,4.63,0.18,98.0,114.0,300.0
3,test,66.16,0.54,12.27,15.77,4.73,0.16,76.0,126.0,314.0
4,control,66.67,0.34,11.77,15.79,4.76,0.17,90.0,117.0,296.0
5,test,66.49,0.55,12.28,15.54,4.64,0.14,81.0,124.0,312.0
6,control,69.46,0.39,9.76,16.16,3.55,0.16,91.0,109.0,286.0
7,test,69.22,0.54,10.38,15.67,3.63,0.15,80.0,120.0,305.0
8,control,68.58,0.33,10.24,16.47,3.75,0.18,101.0,113.0,288.0
9,test,68.75,0.49,10.95,15.57,3.67,0.15,73.0,120.0,309.0
